# 01 — 原始数据探索 + 基准线建立

本 Notebook 完成三件事：
1. **Cell Group A**：探索原始 WARC 数据的结构和分布
2. **Cell Group B**：对比 Trafilatura vs Justext 文本提取器
3. **Cell Group C**：建立所有评估指标的基准线（未过滤数据）

## Cell Group A: 原始数据探索

In [ ]:
import sys, json
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
from pathlib import Path
from collections import Counter
from src.utils.config_loader import load_run_config, print_config_summary

run_cfg = load_run_config()
print_config_summary(run_cfg)
doc_limit = run_cfg['doc_limit']

In [ ]:
# 加载原始数据（优先 JSONL，否则读 WARC）
raw_dir = Path('../data/raw')

# 先尝试找已提取的 JSONL
jsonl_files = list(raw_dir.glob('*.jsonl'))
warc_files = list(raw_dir.glob('*.warc.gz'))

if jsonl_files:
    print(f"读取 JSONL: {jsonl_files[0]}")
    docs = []
    with open(jsonl_files[0]) as f:
        for i, line in enumerate(f):
            if i >= doc_limit:
                break
            try:
                docs.append(json.loads(line))
            except:
                pass
elif warc_files:
    print(f"读取 WARC: {warc_files[0]}")
    import sys
    sys.path.insert(0, '..')
    from src.gen1.pipeline import read_warc_texts
    docs = read_warc_texts(warc_files[0], doc_limit=doc_limit)
else:
    # 使用 FineWeb sample
    parquet_files = list(Path('../data/reference/fineweb_sample').glob('*.parquet'))
    if parquet_files:
        print(f"读取 FineWeb parquet: {parquet_files[0]}")
        df = pd.read_parquet(parquet_files[0])
        docs = [{'text': row['text'], 'url': row.get('url', '')}
                for _, row in df.head(doc_limit).iterrows()]
    else:
        print("⚠️ 未找到数据，使用合成样本演示")
        docs = [{'text': f'This is a sample document number {i}. It contains some text for demonstration.',
                 'url': f'http://example{i}.com/page'} for i in range(100)]

print(f"✅ 加载文档: {len(docs):,} 条")
print(f"字段: {list(docs[0].keys()) if docs else 'N/A'}")

In [ ]:
# 基本统计
texts = [d['text'] for d in docs]
urls = [d.get('url', '') for d in docs]
word_counts = [len(t.split()) for t in texts]
char_counts = [len(t) for t in texts]

print("📊 文档统计:")
print(f"  总文档数: {len(docs):,}")
print(f"  词数 中位数: {np.median(word_counts):.0f} | P10: {np.percentile(word_counts,10):.0f} | P90: {np.percentile(word_counts,90):.0f}")
print(f"  字符数 中位数: {np.median(char_counts):.0f}")
print(f"  总词数: {sum(word_counts):,}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(word_counts, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_xlabel('词数（Words）')
axes[0].set_ylabel('文档数')
axes[0].set_title('文档长度分布（词数）')
axes[0].axvline(np.median(word_counts), color='red', linestyle='--', label=f'中位数={np.median(word_counts):.0f}')
axes[0].legend()

axes[1].hist([min(c, 5000) for c in char_counts], bins=50, color='darkorange', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('字符数（截断至 5000）')
axes[1].set_ylabel('文档数')
axes[1].set_title('文档字符数分布')

plt.suptitle('原始数据基本统计', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/01_raw_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 域名 Top 20
from urllib.parse import urlparse
import re

domains = []
for url in urls:
    try:
        domain = urlparse(url).netloc
        domain = re.sub(r'^www\.', '', domain.lower())
        if domain:
            domains.append(domain)
    except:
        pass

domain_counter = Counter(domains)
top_domains = domain_counter.most_common(20)

fig, ax = plt.subplots(figsize=(12, 5))
domain_names = [d[0][:30] for d in top_domains]
domain_counts = [d[1] for d in top_domains]
bars = ax.barh(range(len(top_domains)), domain_counts, color='steelblue', alpha=0.8)
ax.set_yticks(range(len(top_domains)))
ax.set_yticklabels(domain_names, fontsize=9)
ax.set_xlabel('文档数')
ax.set_title('Top 20 域名分布（原始数据）', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/figures/01_domain_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"唯一域名总数: {len(domain_counter):,}")

In [ ]:
# 展示“脏数据”典型案例
print("=" * 60)
print("  脏数据典型案例（随机采样短文档）")
print("=" * 60)
short_docs = [t for t in texts if len(t.split()) < 20]
import random
random.seed(42)
for i, text in enumerate(random.sample(short_docs[:100], min(5, len(short_docs[:100])))):
    print(f"
案例 {i+1}（{len(text.split())} 词）:")
    print(f"  {repr(text[:200])}")

## Cell Group B: HTML 提取器对比实验

> **为什么提取器的选择至关重要？**
>
> Nemotron-CC 发现从 Justext 换到 Trafilatura 后，高质量 token 数量显著增加。
> 这是一个经常被忽视但影响很大的“上游决策”——
> 在数据进入任何过滤器之前，提取质量就决定了上限。
>
> **两个提取器的核心区别：**
> - **Trafilatura**：专注于正文提取，会分析 HTML 结构（article, main 标签等），过滤导航栏/页脚
> - **Justext**：基于段落级别的统计规则，将段落分类为“near-boilerplate” vs “good content”
>
> **哪个更好？** 取决于数据类型。Trafilatura 在新闻/博客上更准；Justext 在混合内容页面上有时更保守（更少误杀）。FineWeb 和 Nemotron-CC 都选择了 Trafilatura。

In [ ]:
# 提取器对比（需要 trafilatura 和 justext）
# 如果 WARC 文件不存在，跳过此 cell 并用文本说明

try:
    import trafilatura
    import justext
    EXTRACTORS_AVAILABLE = True
except ImportError as e:
    print(f"⚠️ 提取器未安装: {e}")
    print("安装方式: pip install trafilatura justext3")
    EXTRACTORS_AVAILABLE = False

if EXTRACTORS_AVAILABLE and warc_files:
    from warcio.archiveiterator import ArchiveIterator

    html_samples = []
    with open(warc_files[0], 'rb') as f:
        for record in ArchiveIterator(f):
            if record.rec_type == 'response':
                content = record.content_stream().read()
                if b'<html' in content.lower()[:1000]:
                    html_samples.append(content)
                if len(html_samples) >= 20:
                    break

    print(f"收集到 {len(html_samples)} 个 HTML 样本进行对比")

    trafilatura_texts = []
    justext_texts = []

    for html in html_samples:
        # Trafilatura
        t = trafilatura.extract(html, include_tables=False, include_images=False)
        trafilatura_texts.append(t or '')

        # Justext
        try:
            paragraphs = justext.justext(html, justext.get_stoplist('English'))
            j = '
'.join(p.text for p in paragraphs if not p.is_boilerplate)
            justext_texts.append(j)
        except:
            justext_texts.append('')

    # 对比统计
    tra_lens = [len(t.split()) for t in trafilatura_texts]
    jus_lens = [len(t.split()) for t in justext_texts]

    print(f"
📊 提取器对比:")
    print(f"  Trafilatura 平均词数: {np.mean(tra_lens):.0f}")
    print(f"  Justext     平均词数: {np.mean(jus_lens):.0f}")
    print(f"  Trafilatura 空文档率: {sum(1 for t in trafilatura_texts if not t)/len(trafilatura_texts):.1%}")
    print(f"  Justext     空文档率: {sum(1 for t in justext_texts if not t)/len(justext_texts):.1%}")

else:
    print("⚠️ 使用模拟数据展示提取器对比原理")
    print("实际对比需要 WARC 文件 + trafilatura + justext3 库")

## Cell Group C: 建立基准线

> **为什么要在过滤前建立基准线？**
>
> 基准线是衡量每一代 pipeline 效果的零点。
> 如果原始数据 quality_score 均值是 0.30，
> 第一代过滤后是 0.45，第二代是 0.65——
> 这才有意义。没有基准线，“分数提升了”是空话。

In [ ]:
# 建立基准线：对原始数据计算所有评估指标

print("📊 建立基准线（原始数据）...")
print("注意：质量分类器需要先训练，此处展示评估体系的初始化")

from src.evaluation.diversity_metrics import compute_diversity_report

# 多样性基准线（不需要额外模型）
diversity_baseline = compute_diversity_report(
    texts=texts[:run_cfg.get('eval_sample_size', 200)],
    urls=urls[:run_cfg.get('eval_sample_size', 200)],
    sample_size=min(200, len(texts)),
)

print("
📐 多样性基准线:")
for ng_name, ng_stats in diversity_baseline.get('ngram_diversity', {}).items():
    print(f"  {ng_name} unique ratio: {ng_stats.get('unique_ratio', 0):.4f}")

if 'domain_entropy' in diversity_baseline:
    de = diversity_baseline['domain_entropy']
    print(f"  域名 Shannon Entropy: {de.get('entropy', 0):.4f} (归一化: {de.get('normalized_entropy', 0):.4f})")
    print(f"  唯一域名数: {de.get('n_domains', 0):,}")

print(f"
  文档长度中位数: {diversity_baseline.get('length_distribution', {}).get('median', 0):.0f} 词")

In [ ]:
# Token 数量基准线
from src.utils.tokenizer_utils import count_tokens_batch, get_tokenizer

tokenizer = get_tokenizer()
sample_tokens = count_tokens_batch(texts[:100], tokenizer)
avg_tokens = np.mean(sample_tokens)
estimated_total = int(avg_tokens * len(texts))

print(f"📊 Token 基准线（采样 100 条）:")
print(f"  平均 tokens/文档: {avg_tokens:.0f}")
print(f"  估算总 token 数: {estimated_total:,}")
print(f"  估算对应 full_run ({run_cfg.get('doc_limit', 50000):,} 文档): {int(avg_tokens * run_cfg.get('doc_limit', 50000)):,}")

# 保存基准线到 results
import json
baseline = {
    'doc_count': len(docs),
    'avg_tokens_per_doc': float(avg_tokens),
    'estimated_total_tokens': estimated_total,
    'diversity': diversity_baseline,
}
Path('../results/quality_scores').mkdir(parents=True, exist_ok=True)
with open('../results/quality_scores/baseline_metrics.json', 'w') as f:
    json.dump(baseline, f, ensure_ascii=False, indent=2)
print(f"
✅ 基准线已保存: results/quality_scores/baseline_metrics.json")